Setup stuff:

In [9]:
import numpy as np
import pandas as pd
import bambi as bmb
import arviz as az

In [15]:
df = pd.read_csv("domestic_well_radon.csv")
df = df.dropna(subset=["log_radon", "well_depth_ft", "lithology", "aquifer"])
df["log_depth"] = np.log(df["well_depth_ft"])
df["centered_log_depth"] = (df["log_depth"] - df["log_depth"].mean())
df = df[["log_radon", "centered_log_depth", "lithology", "aquifer"]]
df.dropna()
df.head()

,log_radon,centered_log_depth,lithology,aquifer
0,5.950643,-1.004532,Unconsolidated sand and gravel (non-glacial),Surficial
1,5.880533,-0.229892,Unconsolidated sand and gravel (non-glacial),Surficial
2,6.865891,-0.880480,Unconsolidated sand and gravel (non-glacial),Surficial
3,7.259820,-1.103623,Unconsolidated sand and gravel (non-glacial),Surficial
4,7.028201,-1.062801,Unconsolidated sand and gravel (non-glacial),Surficial


Priors (change later):

In [31]:
priors = {
    "Intercept": bmb.Prior("Normal", mu=6.0, sigma=2.0),
    "centered_log_depth": bmb.Prior("Normal", mu=0.0, sigma=1.0),
    "sigma": bmb.Prior("HalfNormal", sigma=1.0),
    "1|lithology": bmb.Prior("Normal", mu=0.0, sigma=bmb.Prior("HalfNormal", sigma=1.0)),
    "1|aquifer": bmb.Prior("Normal", mu=0.0, sigma=bmb.Prior("HalfNormal", sigma=1.0))
}

Model. Hierarchical, with intercept from lithology & aquifer

In [32]:
model = bmb.Model(
    formula = bmb.Formula(
        "log_radon ~ 1 + centered_log_depth + (1 | lithology) + (1 | aquifer)",
    ),
    data=df,
    priors=priors,
)

model_fit = model.fit()

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [sigma, Intercept, centered_log_depth, 1|lithology_sigma, 1|lithology_offset, 1|aquifer_sigma, 1|aquifer_offset]


/home/oscar/school/bda/bda-project/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 8 seconds.
There were 6 divergences after tuning. Increase `target_accept` or reparameterize.


Summary statistics:

In [35]:
print("Divergences:", int(model_fit.sample_stats["diverging"].sum()))
print(
    az.summary(
        model_fit,
        var_names=["Intercept", "centered_log_depth", "sigma", "1|lithology_sigma", "1|aquifer_sigma"],
    )
)

print(az.summary(model_fit, var_names=["1|lithology"]))

Divergences: 6
                       mean     sd eti89_lb eti89_ub ess_bulk ess_tail r_hat mcse_mean  mcse_sd
Intercept               6.1   0.29      5.6      6.6     1750     1954  1.00    0.0069   0.0061
centered_log_depth  -0.1138  0.034    -0.17   -0.059     6197     3317  1.00   0.00044  0.00056
sigma                0.8901  0.014     0.87     0.91     6399     3062  1.00   0.00017  0.00023
1|lithology_sigma      0.78   0.23     0.48      1.2     1646     2083  1.00    0.0057   0.0045
1|aquifer_sigma       0.457  0.074     0.35     0.58     1094     1826  1.00    0.0022   0.0013
                                                                        mean    sd eti89_lb eti89_ub ess_bulk ess_tail r_hat mcse_mean mcse_sd
1|lithology[Basalt and other volcanic-rock aquifers]                   -0.36  0.48     -1.1     0.39     3516     3128  1.00    0.0082  0.0076
1|lithology[Carbonate-rock]                                             -0.5  0.33       -1    0.034     1881     1914  1.0